# 00 — Hello World: look inside a CALOMAPS simulation file

Welcome to CALOMAPS. This notebook is your entry point — it doesn't extract features, doesn't train anything, doesn't reconstruct anything. It just opens **one** simulation output ROOT file, looks at what's inside, and makes a few basic plots so you can see what we're working with.

By the end you should understand:
- where the simulation data lives,
- what an "event" is and what variables are stored per event,
- what a photon shower in the DECAL barrel **looks like**.

**Prerequisites**: `~/setup_calomaps.sh` has been run at least once (so the data dir exists). If you have no simulation data yet, see [`docs/handbook.md`](../docs/handbook.md) §8 to run a smoke-test sim first.

**Kernel**: use `Python (Key4hep)` — the default CVMFS one. We don't need a GPU for any of this.


## 1. Find a data file

In [ ]:
import os
import uproot
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt

# CALOMAPS_DATA_BASE points to where simulation output lives (set by setup_calomaps.sh).
data_base = os.environ.get("CALOMAPS_DATA_BASE", os.path.expanduser("~/CALOMAPS-data"))
dataset_name = "data_spectrum_100um_400GeV"   # the main production dataset
dataset_dir = os.path.join(data_base, dataset_name)

# Pick the first file
import glob
files = sorted(glob.glob(os.path.join(dataset_dir, "sim_photons_part*.root")))
print(f"found {len(files)} files in {dataset_dir}")
print(f"opening first one: {files[0] if files else '(none — generate data first)'}")
filepath = files[0]


## 2. What's inside the file?

The file is a ROOT file. Inside is a `TTree` named `events` (one entry per simulated event), with a flat list of branches that pack everything: truth-particle info, hit positions, hit energies, layer IDs, cell IDs, etc.

`uproot` lets us read these without depending on the C++ ROOT install.


In [ ]:
with uproot.open(filepath) as f:
    print("top-level keys in the file:")
    for k in f.keys()[:8]:
        print(f"  {k}")
    print("...")
    tree = f["events"]
    print(f"\nevents.num_entries = {tree.num_entries}")
    print(f"number of branches  = {len(tree.keys())}")
    print(f"\nfirst 10 branches:")
    for k in tree.keys()[:10]:
        print(f"  {k}")


## 3. Look at a single event

Each event represents one photon entering the DECAL barrel and producing an electromagnetic shower. The shower deposits hits in the silicon pixel layers. The most useful arrays are:

- `MCParticles.*` — the truth-level photon (its energy, direction)
- `ECalBarrelHits/ECalBarrelHits.position.{x,y,z}` — the (x, y, z) of each pixel hit, in mm
- `ECalBarrelHits/ECalBarrelHits.energy` — the energy deposited in each pixel, in GeV

The hit collection is **jagged**: each event has a different number of hits. `awkward` (`ak`) is the library that handles jagged arrays cleanly.


In [ ]:
with uproot.open(filepath) as f:
    tree = f["events"]
    # Read the first event only
    x = tree["ECalBarrelHits/ECalBarrelHits.position.x"].array(entry_stop=1)[0]
    y = tree["ECalBarrelHits/ECalBarrelHits.position.y"].array(entry_stop=1)[0]
    z = tree["ECalBarrelHits/ECalBarrelHits.position.z"].array(entry_stop=1)[0]
    e = tree["ECalBarrelHits/ECalBarrelHits.energy"].array(entry_stop=1)[0]
    # Truth particle: photon's true energy
    p_mc = tree["MCParticles/MCParticles.momentum.y"].array(entry_stop=1)[0]
    print(f"Event 0:")
    print(f"  truth photon momentum.y (= |p| for a +Y beam): {float(p_mc[0]):.2f} GeV")
    print(f"  number of pixel hits: {len(x)}")
    print(f"  hit x range:  [{ak.min(x):.1f}, {ak.max(x):.1f}] mm")
    print(f"  hit y range:  [{ak.min(y):.1f}, {ak.max(y):.1f}] mm   (expect ~1264-1403 for the +Y barrel face)")
    print(f"  hit z range:  [{ak.min(z):.1f}, {ak.max(z):.1f}] mm")
    print(f"  hit energy range: [{ak.min(e)*1e6:.2f}, {ak.max(e)*1e6:.0f}] keV")
    print(f"  total visible E: {float(ak.sum(e)):.4f} GeV   (sampling fraction ~ 15% of truth)")


## 4. Visualize the first event

Two views:
1. **Side profile** (Y vs Z): shows the longitudinal extent of the shower.
2. **Front view** of the silicon layers (X vs Z, with Y as color): shows the lateral shower spread.


In [ ]:
x_np = ak.to_numpy(x)
y_np = ak.to_numpy(y)
z_np = ak.to_numpy(z)
e_np = ak.to_numpy(e)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(z_np, y_np, c=e_np * 1e6, cmap="viridis", s=2, alpha=0.6)
axes[0].set_xlabel("z [mm]")
axes[0].set_ylabel("y [mm]   (radial direction, +Y is the entry face)")
axes[0].set_title("Side profile of one shower")
axes[0].grid(True, alpha=0.3)

# Only hits on the +Y face for the top-down view
front_mask = y_np > 1260
axes[1].scatter(x_np[front_mask], z_np[front_mask],
                c=e_np[front_mask] * 1e6, cmap="viridis", s=2, alpha=0.6)
axes[1].set_xlabel("x [mm]")
axes[1].set_ylabel("z [mm]")
axes[1].set_title("Top-down view of hits on the +Y face")
axes[1].axis("equal")
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(-50, 50)
axes[1].set_ylim(-50, 50)

plt.tight_layout()
plt.show()
print("Each point is one pixel hit. Color = energy deposited in that pixel (keV scale).")


## 5. Aggregate plots across the whole file

Now look at the *distribution* of hits-per-event across all 20 events in this file (each part-file contains 20 events at random energies between 5 and 400 GeV).


In [ ]:
with uproot.open(filepath) as f:
    tree = f["events"]
    x_all = tree["ECalBarrelHits/ECalBarrelHits.position.x"].array()
    e_all = tree["ECalBarrelHits/ECalBarrelHits.energy"].array()
    p_all = tree["MCParticles/MCParticles.momentum.y"].array()

hits_per_event = ak.num(x_all)
total_e_per_event = ak.sum(e_all, axis=1)
true_e_per_event = ak.flatten(p_all)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].hist(true_e_per_event, bins=20)
axes[0].set_xlabel("true photon energy [GeV]")
axes[0].set_ylabel("events")
axes[0].set_title("Truth energy spectrum")
axes[0].grid(True, alpha=0.3)

axes[1].scatter(true_e_per_event, hits_per_event, s=20)
axes[1].set_xlabel("true photon energy [GeV]")
axes[1].set_ylabel("number of pixel hits")
axes[1].set_title("Hits vs E_true (digital readout)")
axes[1].grid(True, alpha=0.3)

axes[2].scatter(true_e_per_event, total_e_per_event, s=20)
axes[2].plot([0, 400], [0, 60], "k--", alpha=0.5, label="15% sampling fraction")
axes[2].set_xlabel("true photon energy [GeV]")
axes[2].set_ylabel("visible energy [GeV]")
axes[2].set_title("E_visible vs E_true (analog readout)")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 6. What you just saw — and where to next

In 20 events at random energies 5-400 GeV:

- Higher-energy photons make more pixel hits (digital readout looks roughly linear in E).
- The visible energy is ~15% of the true energy — that's the **sampling fraction** of a tungsten-silicon calorimeter (most of the shower energy goes into the tungsten absorber, only a fraction is detected in the silicon).
- Each individual shower has thousands of hits, clustered transversely around the beam axis.

What we **haven't** seen yet:
- The crossover where digital readout starts losing to analog as energy goes up (pixel saturation). To see that, you need to aggregate over many events at many energies — see [`01_geometry_visualization.ipynb`](01_geometry_visualization.ipynb) for the calorimeter geometry and [`02_data_extraction.ipynb`](02_data_extraction.ipynb) for the heavy parallel extraction.
- A trained surrogate model that maps E_true to E_visible (with uncertainty). That's in [`03_ml_training_and_eval.ipynb`](03_ml_training_and_eval.ipynb).

For physics context, see [`docs/DECAL_pipeline.md`](../docs/DECAL_pipeline.md) and [`docs/handbook.md`](../docs/handbook.md).
